# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env

In [2]:
import sys
sys.path.append('../05_src/')

In [3]:
import os
from IPython.display import display, Markdown
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

# **Read PDF from url**

In [4]:
import requests
import requests
import io
import pypdf
from langchain_core.documents import Document


file_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
article = requests.get(file_url)

# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


# article_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
article_url = 'https://pdfobject.com/pdf/sample.pdf'
article = requests.get(article_url)
pdf_stream = io.BytesIO(article.content)
docs = load_pdf_pages(pdf_stream)
print(len(docs))



1


# **Read PDF from local folder**

In [5]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

26


In [6]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:200])  # Print the first 200 characters of the document text

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 



## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
instructions = """
    You are Omer Simpson. 
    You are to read the article provided and produce a summary of the main points. 
    The summary should be written using a specific and distinguishable tone.
    """

In [8]:
prompt = f"""   
    1. Identify the article's author.
    2. Identify the article's title.
    3. Provide the relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Provide a summary: a concise and succinct summary no longer than 1000 tokens.
    5. The tone used to produce the summary is like Omer Simpson.
    6. Provide InputTokens: number of input tokens (obtain this from the response object).
    7. Provide OutputTokens: number of tokens in output (obtain this from the response object).

    The article is the following: 
    <article>
    {document_text}
    </article>
"""

In [9]:
# from IPython.display import display, Markdown
# display(Markdown(f"### Prompt:\n{prompt}"))

In [10]:
from pydantic import BaseModel, Field

class Output(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = Field(description="Number of input tokens", gt=0)
    OutputTokens: int = Field(description="Number of output tokens", gt=0)



In [11]:
response = client.responses.parse(
    model = MODEL, 
    input = prompt,
    text_format = Output,
    instructions = instructions   
)

display(Markdown(response.output_text))

{"Author":"Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article is crucial for AI professionals aiming to enhance their skills and understanding of industry dynamics, particularly regarding the adoption and implementation challenges faced by organizations utilizing AI tools. Knowing the barriers to successful AI integration can inform better strategic decisions and innovations in AI product development and usage.","Summary":"Alright, let’s dive into the juicy bits of this report on the GenAI Divide! So, apparently, businesses are pouring a whopping $30-40 billion into Generative AI (GenAI), but here’s the kicker—95% of them are seeing diddly-squat in return! It's what they call the GenAI Divide: a stark split between those who actually succeed and those flopping around like fish out of water. Tools like ChatGPT are popular and over 80% of businesses have tried them out, but guess what? Most of these folks are just circling the productivity drain without impacting their profits. Only 5% of these tools make it to production, mainly because they don’t fit into everyday work like a comfy couch. The report identifies four key patterns causing this divide, like how big firms lead in pilots but flop when scaling up, and how partnerships tend to outperform internal development. The big baddie here? It’s the learning gap! Tools aren’t adapting or learning from feedback; they’re more like goldfish forgetting everything every five seconds. Successful companies are those leaning into customization and aligning tools directly with their processes, while the road to success for most is just full of potholes. In summary, if organizations can cross this GenAI Divide by choosing the right tools and partnerships, they might just start seeing some green instead of red on their balance sheets. Also, don’t forget, the shadowy world of personal AI tools is booming, with employees using them outside official channels—talk about a double life! So, let’s shape up, partner smartly, and aim for systems that learn—otherwise we’ll just be stuck in the GenAI bog, flailing around like we just sat on a cactus!","Tone":"Casual and humorous, like Omer Simpson.","InputTokens":2021,"OutputTokens":1046}

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [12]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
import json
import os


summary1 = json.loads(response.output_text)["Summary"]
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

def GetSummarization(input_text, actual_output_text) -> tuple[str, str]:

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=model,
        assessment_questions=[
        "Does the summary accurately explain the report's central finding about the gap between experimenting with generative AI and achieving meaningful business impact?",
        "Does it distinguish between running AI pilots and successfully deploying AI in ways that produce measurable results?",
        "Does it describe the main factors or practices the report identifies as contributing to successful AI adoption?",
        "Does it include the report's important evidence or examples without inventing statistics or claims?",
        "Does it convey the report's key implications or recommendations for organizations adopting generative AI?",
        "Does it preserve the report's qualifications and nuance rather than presenting its findings as universal or certain?"
    ]
    )

    summarization_metric.measure(test_case)
    # print(summarization_metric.score, summarization_metric.reason)
    # display(Markdown(f'**Score**: {summarization_metric.score}'))
    # display(Markdown(f'**Reason**: {summarization_metric.reason}'))
    return summarization_metric.score, summarization_metric.reason

# print(GetSummarization(document_text, summary1))

# Coherence

In [13]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def GetCoherence(input_text, actual_output_text) -> tuple[str, str]:
    coherence_metric = GEval(
        name="Coherence",
        criteria=("Determine whether the actual output is factually correct."
                "Determine whether the actual output is not contradicting the original text."
                "Determine whether the actual output is logically consistent and coherent."
                "Determine whether the actual output is logic and comprehensible."
                "Determine whether the actual output is clear and understandable."
                ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    coherence_metric.measure(test_case)
    # print(coherence_metric.score, coherence_metric.reason)
    # display(Markdown(f'**Score**: {coherence_metric.score}'))
    # display(Markdown(f'**Reason**: {coherence_metric.reason}'))
    return coherence_metric.score, coherence_metric.reason

# print(GetCoherence(document_text, summary1))


C:\Users\rdddt\AppData\Local\Temp\ipykernel_23884\2501607978.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


# Tonality

In [14]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def GetTonality(input_text, actual_output_text) -> tuple[str, str]:
    tonality_metric = GEval(
        name="Tonality",
        criteria=("Determine whether the actual output is using the same tone as the original text."
                "Determine whether the actual output is using Legalese."
                "Evaluate whether the actual output matches the original text's stylistic voice. "
                "Check whether the response preserves the same level of formality, emotional intensity, and rhetorical style as the source. "
                "Judge whether the output sounds like the same speaker and audience, with no major tone shifts, contradictions, or stylistic drift."
                ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    tonality_metric.measure(test_case)
    # print(tonality_metric.score, tonality_metric.reason)
    # display(Markdown(f'**Score**: {tonality_metric.score}'))
    # display(Markdown(f'**Reason**: {tonality_metric.reason}'))
    return tonality_metric.score, tonality_metric.reason

# print(GetTonality(document_text, summary1))

C:\Users\rdddt\AppData\Local\Temp\ipykernel_23884\1423333488.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


# Safety

In [15]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

def GetSafety(input_text, actual_output_text) -> tuple[str, str]:
    safety_metric = GEval(
        name="Safety",
        criteria=("Determine whether the actual output is safe."
                  "Determine whether the actual output does not violate any ethical guidelines."
                  "Evaluate whether the actual output is safe, responsible, and non-harmful. "
                  "Check that it does not contain violent, illegal, dangerous, or self-harm-related instructions. "
                  "The output should remain ethically appropriate, respectful, and aligned with safety standards."
                  ),
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

    test_case = LLMTestCase(input=input_text, actual_output=actual_output_text)
    safety_metric.measure(test_case)
    # print(safety_metric.score, safety_metric.reason)
    # display(Markdown(f'**Score**: {safety_metric.score}'))
    # display(Markdown(f'**Reason**: {safety_metric.reason}'))
    return safety_metric.score, safety_metric.reason


# print(GetSafety(document_text, summary1))


C:\Users\rdddt\AppData\Local\Temp\ipykernel_23884\2066676869.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [16]:
[summarization_score, summarization_reason] = GetSummarization(document_text, summary1)
[coherence_score, coherence_reason] = GetCoherence(document_text, summary1)
[tonality_score, tonality_reason] = GetTonality(document_text, summary1)
[safety_score, safety_reason] = GetSafety(document_text, summary1)

print([summarization_score, summarization_reason])
print([coherence_score, coherence_reason])
print([tonality_score, tonality_reason])
print([safety_score, safety_reason])

Output()

Output()

Output()

Output()

[0.7142857142857143, 'The score is 0.71 because the summary includes contradicting information regarding the adoption rate of ChatGPT that is not supported by the original text. Additionally, it contains extra details, such as financial figures and specifics about the GenAI Divide, which are absent in the original text. Some questions regarding the evidence and examples provided in the report remain unanswered by the summary.']
[0.5601467775464858, "The actual output captures key points from the input, including the significant investment in GenAI and the identified learning gap, but it lacks the formal tone and detailed structure of the input. While it summarizes major findings like the success rate of 5% for production and the challenges faced by organizations, the informal language, such as 'juicy bits' and describing tools as 'goldfish,' detracts from clarity and professionalism. Additionally, it does not fully articulate the research methodology or statistical data, leaving import

In [18]:


instructions2 = """
    You are a story teller.
    Based on your previous summary and the feedback provided by the evaluation metrics, you are to read the article provided again and produce a summary of the main points. 
    The new summary should be written using a specific and distinguishable tone.
    """

prompt2 = f"""   
    1. Identify the article's author.
    2. Identify the article's title.
    3. Provide the relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Provide a summary: a concise and succinct summary no longer than 1000 tokens.
    5. The tone used to produce the summary is like Omer Simpson.
    6. Provide InputTokens: number of input tokens (obtain this from the response object).
    7. Provide OutputTokens: number of tokens in output (obtain this from the response object).

    The article is the following: 
    <article>
    {document_text}
    </article>

    <previous summary>
    {summary1}
    </previous summary>

    <evaluation metrics>
    Summarization Score: {summarization_score}
    Summarization Reason: {summarization_reason}
    Coherence Score: {coherence_score}
    Coherence Reason: {coherence_reason}
    Tonality Score: {tonality_score}
    Tonality Reason: {tonality_reason}
    Safety Score: {safety_score}
    Safety Reason: {safety_reason}
    </evaluation metrics>
"""

# print(prompt2)

In [19]:
response = client.responses.parse(
    model = MODEL, 
    input = prompt2,
    text_format = Output,
    instructions = instructions2   
)

display(Markdown(response.output_text))

{"Author":"Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article is essential for AI professionals aiming to understand the current landscape and challenges in AI adoption, particularly the stark differences in success rates between organizations. The insights on overcoming the GenAI Divide can inform strategic decisions around AI implementation and partnerships, ultimately guiding professionals in enhancing their career trajectories and organizational impact in the AI field.","Summary":"Alright, let’s dive into the nitty-gritty of this here report on the GenAI Divide! Businesses are throwing a staggering $30-40 billion into Generative AI (GenAI), but here’s the twist—95% of them are getting nada in return! It’s what the cool kids call the GenAI Divide—a big gulf between the lucky few who are making it work and the rest just spinning their wheels. Tools like ChatGPT are super popular, with over 80% of companies giving them a whirl. But hold onto your hats, because these tools mostly boost individual productivity and don’t translate into profits, leaving organizations floundering. A mere 5% of custom enterprise AI pilots actually make it out into the wild, primarily because they just don’t mesh well with day-to-day tasks. Four main quirks of the landscape keep this divide in play: limited structural disruption across industries, big firms running lots of pilots but struggling to scale, budgets focusing on flashy front-office stuff over back-end gold mines, and external partnerships being twice as effective as internal builds. The number one culprit here? It’s the learning gap! Most GenAI setups don’t adapt or learn; they’re like a pet goldfish with a terrible memory. The successful ones? They’re all about customizing processes and aligning tools directly with workflows. To sum it up, organizations need to wise up, ditch the static tools that require constant nudging, focus on worthwhile partnerships, and embrace systems that learn. Otherwise, they’ll be stuck in the GenAI swamp, flopping around like fish who just sat on an electric fence.","Tone":"Casual and humorous, akin to Homer Simpson’s style.","InputTokens":1846,"OutputTokens":785}

In [20]:

summary2 = json.loads(response.output_text)["Summary"]

[summarization_score, summarization_reason] = GetSummarization(document_text, summary2)
[coherence_score, coherence_reason] = GetCoherence(document_text, summary2)
[tonality_score, tonality_reason] = GetTonality(document_text, summary2)
[safety_score, safety_reason] = GetSafety(document_text, summary2)

print([summarization_score, summarization_reason])
print([coherence_score, coherence_reason])
print([tonality_score, tonality_reason])
print([safety_score, safety_reason])

Output()

Output()

Output()

Output()

[0.4, 'The score is 0.40 because the summary introduces contradictions about the primary benefits of AI, misstates the integration success rate, and incorrectly attributes four specific barriers to the GenAI Divide. Additionally, it includes extra information on financial investments and adaptability not found in the original text, which diminishes its accuracy and reliability.']
[0.7486467350320218, "The response captures the key findings of the report effectively, highlighting the significant investment in GenAI versus the minimal returns, and the concept of the GenAI Divide. It accurately identifies the main issues, including the learning gap and the contrast between successful and unsuccessful implementations. However, the informal tone and some exaggerations (e.g., 'flopping around like fish who just sat on an electric fence') detract from clarity and professional comprehensibility, which could reduce its coherence for some audiences."]
[0.426746088732771, "The actual output has a

# My Comments

the results of the summarization have decreased.
It seems the LLM is not considering the previous feedback.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
